In [44]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid", context="talk")

ancovas_mean_results_path = '../../../../results/pet/paper_analysis/ancova_rois_mean/'
ancovas_cerebellum_results_path = '../../../../results/pet/paper_analysis/ancova_rois_cerebellum/'
neurocloud_results_path = '../../../../results/pet/paper_analysis/neurocloud/'
rois_to_lobe_csv = os.path.join('../../../../data/pet/', 'hammers_roi_to_lobe.csv')
rois_abbreviations_csv = os.path.join('../../../../data/pet/', 'roi_abbreviations.csv')
output_path = os.path.join('../../../../results/pet/paper_analysis/', 'consensus/')
os.makedirs(output_path, exist_ok=True)

# Load rois to lobe mapping
rois_to_lobe_df = pd.read_csv(rois_to_lobe_csv)
# Load abbreviations for ROIs
rois_abbreviations_df = pd.read_csv(rois_abbreviations_csv)


In [45]:
# Perform consensus analysis between ceunim controls, cermep and neurocloud
csv_ceunim_controls_filename = os.path.join(ancovas_mean_results_path, 'ancova_lc_vs_ceunim_controls_results_time_to_acquisition.csv')
csv_cermep_filename = os.path.join(ancovas_mean_results_path, 'ancova_lc_vs_cermep_controls_results.csv')
csv_neurocloud_filename = os.path.join(neurocloud_results_path, 'ttest_neurocloud_long_covid_hammers_names.csv')  
# Read each csv
ceunim_controls_df = pd.read_csv(csv_ceunim_controls_filename)
cermep_df = pd.read_csv(csv_cermep_filename)
neurocloud_df = pd.read_csv(csv_neurocloud_filename)

# Now we identify:
# First, the ROIs that p_FDR<=0.05 for the three datasets
# For cerunim and cermep the fiels are: ROI,n,F_group,p_group,beta_group,mean_control,mean_lc,delta_lc_minus_control,direction,p_sex,beta_sex,p_age,beta_age,p_fdr,significant_fdr
# For neurocloud the fields are:Region,N,MEAN_Z,MEDIAN_Z,STD_Z,COHEN_D,TSTAT,PVAL_T,WSTAT,PVAL_W,Effect, P_FDR
ceunim_controls_significant_rois = ceunim_controls_df[ceunim_controls_df['p_fdr'] <= 0.05]['ROI']
cermep_significant_rois = cermep_df[cermep_df['p_fdr'] <= 0.05]['ROI']
neurocloud_significant_rois = neurocloud_df[neurocloud_df['PVAL_FDR'] <= 0.05]['Region']

consensus_rois = set(ceunim_controls_significant_rois) & set(cermep_significant_rois) & set(neurocloud_significant_rois)
# and a second condition is that the direction is the same across the three datasets, therefore mean_lc and MEAN_Z have the same sign
consensus_rois_with_same_direction = set()
for roi in consensus_rois:
    mean_lc_ceunim = ceunim_controls_df.loc[ceunim_controls_df['ROI'] == roi, 'mean_lc'].values[0]
    mean_lc_cermep = cermep_df.loc[cermep_df['ROI'] == roi, 'mean_lc'].values[0]
    mean_z_neurocloud = neurocloud_df.loc[neurocloud_df['Region'] == roi, 'MEAN_Z'].values[0]
    if (mean_lc_ceunim > 0 and mean_lc_cermep > 0 and mean_z_neurocloud > 0) or (mean_lc_ceunim < 0 and mean_lc_cermep < 0 and mean_z_neurocloud < 0):
        consensus_rois_with_same_direction.add(roi)
print("Consensus ROIs with the same direction across all three datasets:")
print(consensus_rois_with_same_direction)

# Create a df with the consensus ROIs, the p_values, p_FDR and effect sizes and their directions
consensus_data = []
for roi in consensus_rois_with_same_direction:
    ceunim_row = ceunim_controls_df[ceunim_controls_df['ROI'] == roi].iloc[0]
    cermep_row = cermep_df[cermep_df['ROI'] == roi].iloc[0]
    neurocloud_row = neurocloud_df[neurocloud_df['Region'] == roi].iloc[0]
    consensus_data.append({
        'ROI': roi,
        'ceunim_p': ceunim_row['p_group'],
        'ceunim_p_fdr': ceunim_row['p_fdr'],
        'ceunim_mean_lc': ceunim_row['mean_lc'],
        'cermep_p': cermep_row['p_group'],
        'cermep_p_fdr': cermep_row['p_fdr'],
        'cermep_mean_lc': cermep_row['mean_lc'],
        'neurocloud_p': neurocloud_row['PVAL_T'],
        'neurocloud_p_fdr': neurocloud_row['PVAL_FDR'],
        'neurocloud_mean_z': neurocloud_row['MEAN_Z'],
        'ceunim_beta': ceunim_row['beta_group'],
        'ceunim_effect': ceunim_row['delta_lc_minus_control'],
        'ceunim_direction': 'positive' if ceunim_row['delta_lc_minus_control'] > 0 else 'negative',
        'cermep_beta': cermep_row['beta_group'],
        'cermep_effect': cermep_row['delta_lc_minus_control'],
        'cermep_direction': 'positive' if cermep_row['delta_lc_minus_control'] > 0 else 'negative',
        'neurocloud_effect': neurocloud_row['MEAN_Z'],
        'neurocloud_direction': 'positive' if neurocloud_row['MEAN_Z'] > 0 else 'negative'
    })
consensus_df = pd.DataFrame(consensus_data)
print("Consensus ROIs with detailed statistics:")
print(consensus_df)
consensus_df.to_csv(os.path.join(output_path, "consensus_rois.csv"), index=False)




Consensus ROIs with the same direction across all three datasets:
{'FL-superior-frontal-gyrus-L', 'FL-inferior-frontal-gyrus-L', 'CG-anterior-cingulate-gyrus-R', 'FL-middle-frontal-gyrus-L', 'CG-posterior-cingulate-gyrus-L', 'PL-supramarginal-gyrus-L', 'FL-posterior-orbital-gyrus-L', 'PL-supramarginal-gyrus-R', 'FL-lateral-orbital-gyrus-L', 'FL-superior-frontal-gyrus-R', 'FL-inferior-frontal-gyrus-R', 'PL-angular-gyrus-L', 'FL-subgenual-frontal-cortex-R', 'insula-anterior-short-gyrus-R'}
Consensus ROIs with detailed statistics:
                               ROI  ceunim_p  ceunim_p_fdr  ceunim_mean_lc  \
0      FL-superior-frontal-gyrus-L  0.005523      0.039774        5.136589   
1      FL-inferior-frontal-gyrus-L  0.000149      0.002746        5.601804   
2    CG-anterior-cingulate-gyrus-R  0.000849      0.011161        4.747051   
3        FL-middle-frontal-gyrus-L  0.000428      0.006563        5.677183   
4   CG-posterior-cingulate-gyrus-L  0.011133      0.047962        5.854607  

In [46]:
# Now create a list of those which are significant in at least two datasets
# First keep ROIs with the same direction across datasets
consensus_at_least_two_datasets_rois = set()
for roi in ceunim_controls_df['ROI'].values:
    mean_lc_ceunim = ceunim_controls_df.loc[ceunim_controls_df['ROI'] == roi, 'mean_lc'].values[0]
    mean_lc_cermep = cermep_df.loc[cermep_df['ROI'] == roi, 'mean_lc'].values[0]
    if roi in neurocloud_df['Region'].values:
        mean_z_neurocloud = neurocloud_df.loc[neurocloud_df['Region'] == roi, 'MEAN_Z'].values[0]
    else:
        mean_z_neurocloud = 0.01*np.sign(mean_lc_ceunim) # if we set it to None wont comply with the condition but we want to keep it inc ase cermep and ceunim have the same direction
    if (mean_lc_ceunim > 0 and mean_lc_cermep > 0 and mean_z_neurocloud > 0) or (mean_lc_ceunim < 0 and mean_lc_cermep < 0 and mean_z_neurocloud < 0):
        consensus_at_least_two_datasets_rois.add(roi)

from collections import Counter
all_significant_rois = list(ceunim_controls_significant_rois) + list(cermep_significant_rois) + list(neurocloud_significant_rois)
print(all_significant_rois)
roi_counts = Counter(all_significant_rois)
at_least_two_datasets_rois = {roi for roi, count in roi_counts.items() if count >= 2}
print("ROIs significant in at least two datasets:")
print(at_least_two_datasets_rois)

# Now from the ROIs with the same direction, keep only those that are significant in at least two datasets
consensus_at_least_two_datasets_rois = consensus_at_least_two_datasets_rois.intersection(at_least_two_datasets_rois)
print("Consensus ROIs with the same direction and significant in at least two datasets:")
print(consensus_at_least_two_datasets_rois)



# Create a df with the ROIs significant in at least two datasets
at_least_two_datasets_data = []
for roi in at_least_two_datasets_rois:
    ceunim_row = ceunim_controls_df[ceunim_controls_df['ROI'] == roi].iloc[0]
    cermep_row = cermep_df[cermep_df['ROI'] == roi].iloc[0]
    # neurocloud may be missiing a roin,s o check that exists first, if not leave empty
    if roi in neurocloud_df['Region'].values:
        neurocloud_row = neurocloud_df[neurocloud_df['Region'] == roi].iloc[0]
        neurocloud_direction = 'positive' if neurocloud_row['MEAN_Z'] > 0 else 'negative'
    else:
        neurocloud_row = pd.Series({'PVAL_T': None, 'MEAN_Z': None, 'PVAL_FDR': None, 'neurocloud_direction': None})
    at_least_two_datasets_data.append({
         'ROI': roi,
                'ceunim_p': ceunim_row['p_group'],
                'ceunim_p_fdr': ceunim_row['p_fdr'],
                'ceunim_mean_lc': ceunim_row['mean_lc'],
                'cermep_p': cermep_row['p_group'],
                'cermep_p_fdr': cermep_row['p_fdr'],
                'cermep_mean_lc': cermep_row['mean_lc'],
                'neurocloud_p': neurocloud_row['PVAL_T'],
                'neurocloud_p_fdr': neurocloud_row['PVAL_FDR'],
                'neurocloud_mean_z': neurocloud_row['MEAN_Z'],
                'ceunim_beta': ceunim_row['beta_group'],
                'ceunim_effect': ceunim_row['delta_lc_minus_control'],
                'ceunim_direction': 'positive' if ceunim_row['delta_lc_minus_control'] > 0 else 'negative',
                'cermep_beta': cermep_row['beta_group'],
                'cermep_effect': cermep_row['delta_lc_minus_control'],
                'cermep_direction': 'positive' if cermep_row['delta_lc_minus_control'] > 0 else 'negative',
                'neurocloud_effect': neurocloud_row['MEAN_Z'],
                'neurocloud_direction': neurocloud_direction
    })
at_least_two_datasets_df = pd.DataFrame(at_least_two_datasets_data)
print("ROIs significant in at least two datasets with detailed statistics:")
print(at_least_two_datasets_df)

# Save them in csv files
at_least_two_datasets_df.to_csv(os.path.join(output_path, "consensus_at_least_two_datasets_rois.csv"), index=False)

['TL-middle-and-inferior-temporal-gyrus-L', 'PL-supramarginal-gyrus-L', 'FL-lateral-orbital-gyrus-L', 'TL-posterior-temporal-lobe-L', 'FL-inferior-frontal-gyrus-L', 'FL-middle-frontal-gyrus-L', 'CG-anterior-cingulate-gyrus-R', 'FL-superior-frontal-gyrus-R', 'FL-pre-subgenual-frontal-cortex-R', 'FL-posterior-orbital-gyrus-L', 'TL-middle-and-inferior-temporal-gyrus-R', 'FL-superior-frontal-gyrus-L', 'TL-anterior-temporal-lobe-lateral-part-L', 'cerebellum-R', 'PL-angular-gyrus-L', 'insula-anterior-long-gyrus-R', 'pallidum-L', 'insula-anterior-short-gyrus-R', 'PL-supramarginal-gyrus-R', 'CG-posterior-cingulate-gyrus-L', 'FL-subgenual-frontal-cortex-R', 'FL-inferior-frontal-gyrus-R', 'FL-anterior-orbital-gyrus-L', 'PL-supramarginal-gyrus-L', 'FL-lateral-orbital-gyrus-L', 'FL-middle-frontal-gyrus-L', 'TL-posterior-temporal-lobe-L', 'TL-middle-and-inferior-temporal-gyrus-L', 'FL-inferior-frontal-gyrus-L', 'brainstem-excluding-substantia-nigra-', 'CG-anterior-cingulate-gyrus-R', 'FL-superior-f